# Real VIIRS swath → GeoBox: `odc-geo` vs. Satpy vs. FAISS nearest neighbour

This notebook is a follow-up to `06-swath-to-geobox-odc-vs-faiss.ipynb` and uses the same three reprojection strategies, but on a **real VIIRS L1B swath** from NASA Earthdata (collections `VJ202IMG` / `VJ203IMG`, band `I04`).

It mirrors the `02-viirs.ipynb` pipeline:

1. Search NASA Earthdata with `aereo.builtins.search.search_earthaccess`.
2. Build extraction tasks with the VIIRS grid/patch config.
3. Read one task with `aereo.read_satpy.read_satpy`.
4. Reproject the cropped 2-D lat/lon swath to the patch's UTM GeoBox with:
   - `odc.geo.xr.xr_reproject` (after a quick nearest-neighbour rectification to a regular lat/lon grid)
   - `satpy.Scene.resample` (via `aereo.reproject_satpy`)
   - FAISS nearest neighbour

> You need a NASA Earthdata login. `earthaccess` will look for credentials in `~/.netrc` or prompt you interactively.

> **Why the intermediate lat/lon grid for `odc-geo`?** `xr_reproject` needs a rectilinear source. A real VIIRS scan is highly curved, so a low-order GCP polynomial (the only direct way to feed 2-D lat/lon into `odc-geo`) cannot follow the scan geometry and produces a visibly different image. Rectifying the swath to a regular lat/lon grid first gives `odc-geo` a source it can reproject accurately.

## Optional dependencies

In [ ]:
# Uncomment if these are not already installed.
# %pip install faiss-cpu earthaccess scipy

## Imports

In [ ]:
import time

import faiss
import matplotlib.pyplot as plt
import numpy as np
import pyproj
import xarray as xr
from odc.geo.xr import xr_coords, xr_reproject
from scipy.interpolate import griddata

from aereo.asset_downloader.core import _download_with_earthaccess
from aereo.builtins.search import search_earthaccess
from aereo.builtins.task_builder import build_grouped_tasks
from aereo.pipeline import ExtractionJob
from aereo.read_satpy import read_satpy
from aereo.reproject_satpy import reproject_satpy

## 1. Search and build tasks

In [ ]:
from datetime import datetime, timezone

# Load the same VIIRS job used in 02-viirs.ipynb.
job = ExtractionJob.load_from_config(
    config_dir="config",
    config_name="job_viirs",
)

assets = search_earthaccess(
    collections={"VJ202IMG": ["I04"], "VJ203IMG": []},
    intersects="config/aoi/chocon.geojson",
    start_datetime=datetime(2024, 1, 1, tzinfo=timezone.utc),
    end_datetime=datetime(2024, 1, 1, 23, 59, 59, tzinfo=timezone.utc),
)
print(f"✓ Found {len(assets)} VIIRS granules")

tasks = build_grouped_tasks(
    search_results=assets,
    job=job,
    cells_per_task=5,
)
print(f"✓ Built {len(tasks)} tasks")

# Pick the first task and its first patch.
task = tasks[0]
patch = task.patches[0]
dst_geobox = patch.geobox
print(f"Target GeoBox: {dst_geobox}")

## 2. Read the VIIRS swath with Satpy

In [ ]:
# Download the task assets and load I04 with Satpy.
# This returns an xarray.Dataset with 2-D latitude/longitude coordinates.
ds = read_satpy(
    task,
    reader="viirs_l1b",
    wishlist=["I04"],
    download_workers=1,
    downloader=_download_with_earthaccess,
)

# Load into memory so downstream reprojections are fast.
ds = ds.load()

da = ds["I04"]
lats = da.coords["latitude"].values
lons = da.coords["longitude"].values
values = da.values.astype("float32")
fill_value = da.attrs.get("_FillValue", np.nan)
valid_mask = ~np.isnan(values) if np.isnan(fill_value) else values != fill_value

print(f"Swath shape: {da.shape}")
print(f"Lat range:   {lats.min():.4f} → {lats.max():.4f}")
print(f"Lon range:   {lons.min():.4f} → {lons.max():.4f}")
print(f"Data range:  {values[valid_mask].min():.2f} → {values[valid_mask].max():.2f}")
print(f"Fill value:  {fill_value}")

## 3. Method 1 — `odc-geo` reproject

In [ ]:
# odc-geo's xr_reproject needs a rectilinear source. First rectify the swath to a
# temporary regular lat/lon grid using nearest-neighbour interpolation, then let
# odc-geo warp that grid to the target UTM GeoBox.

# Target patch bounds in lat/lon (with a small buffer).
inv_transformer = pyproj.Transformer.from_crs(
    dst_geobox.crs, "EPSG:4326", always_xy=True
)
tgt_y = dst_geobox.coordinates["y"].values
tgt_x = dst_geobox.coordinates["x"].values
tgt_yy, tgt_xx = np.meshgrid(tgt_y, tgt_x, indexing="ij")
tgt_lon, tgt_lat = inv_transformer.transform(tgt_xx, tgt_yy)

buffer = 0.1  # degrees
lon_min = float(tgt_lon.min() - buffer)
lon_max = float(tgt_lon.max() + buffer)
lat_min = float(tgt_lat.min() - buffer)
lat_max = float(tgt_lat.max() + buffer)

# Regular lat/lon grid at roughly the source resolution (~375 m → ~0.0035°).
grid_res = 0.005  # degrees
lon_grid = np.arange(lon_min, lon_max, grid_res)
lat_grid = np.arange(lat_min, lat_max, grid_res)
lon_g, lat_g = np.meshgrid(lon_grid, lat_grid)

# Nearest-neighbour rectification.
t0 = time.perf_counter()
regular_values = griddata(
    (lons[valid_mask], lats[valid_mask]),
    values[valid_mask],
    (lon_g, lat_g),
    method="nearest",
)
regular_da = xr.DataArray(
    regular_values,
    dims=("y", "x"),
    coords={"y": lat_grid, "x": lon_grid},
    name="I04_regular",
)
regular_da = regular_da.odc.assign_crs("EPSG:4326")

# Reproject the regular grid to the patch's UTM GeoBox.
odc_da = xr_reproject(regular_da, dst_geobox, resampling="nearest", dst_nodata=np.nan)
odc_time = time.perf_counter() - t0
print(f"odc-geo reproject took {odc_time:.3f} s")

## 4. Method 2 — Satpy `Scene.resample`

In [ ]:
# Use the built-in aereo Satpy reprojector. It reconstructs a Satpy Scene from
# the dataset's `area` attributes and resamples to an AreaDefinition derived
# from each patch's GeoBox.
t0 = time.perf_counter()
satpy_outputs = reproject_satpy(ds, task)
satpy_time = time.perf_counter() - t0

satpy_da = satpy_outputs[patch.id]["I04"]
print(f"Satpy resample took {satpy_time:.3f} s")
print(f"Satpy result shape: {satpy_da.shape}")

## 5. Method 3 — FAISS nearest neighbour

In [ ]:
# Flatten the valid swath pixels into a 2-D (lon, lat) point cloud.
src_pts = np.column_stack([lons[valid_mask], lats[valid_mask]]).astype("float32")

# Exact L2 nearest-neighbour index in (lon, lat) space.
index = faiss.IndexFlatL2(src_pts.shape[1])
index.add(src_pts)

# Target grid centre coordinates in UTM → back to lat/lon for the NN query.
tgt_lon_q, tgt_lat_q = inv_transformer.transform(tgt_xx.ravel(), tgt_yy.ravel())
tgt_pts = np.column_stack([tgt_lon_q, tgt_lat_q]).astype("float32")

# Query.
t0 = time.perf_counter()
distances, nearest = index.search(tgt_pts, 1)
faiss_time = time.perf_counter() - t0

# Reshape to the target grid.
src_flat = values[valid_mask]
faiss_values = src_flat[nearest.ravel()].reshape(dst_geobox.shape.yx)
faiss_da = xr.DataArray(
    faiss_values,
    dims=("y", "x"),
    coords=xr_coords(dst_geobox),
    name="faiss_nearest",
)
print(f"FAISS nearest neighbour took {faiss_time:.3f} s")

## 6. Compare results

In [ ]:
common = (
    np.isfinite(odc_da.values)
    & np.isfinite(satpy_da.values)
    & np.isfinite(faiss_da.values)
)

# Pairwise differences.
odc_sat = np.abs(odc_da.values - satpy_da.values)
odc_fai = np.abs(odc_da.values - faiss_da.values)
sat_fai = np.abs(satpy_da.values - faiss_da.values)

print(f"Common valid pixels: {common.sum():,}")
print()
print(f"odc-geo time:  {odc_time:.3f} s")
print(f"Satpy time:    {satpy_time:.3f} s")
print(f"FAISS time:    {faiss_time:.3f} s")
print()
print("Mean absolute differences (K):")
print(f"  odc-geo  vs Satpy:   {odc_sat[common].mean():.3f}")
print(f"  odc-geo  vs FAISS:   {odc_fai[common].mean():.3f}")
print(f"  Satpy    vs FAISS:   {sat_fai[common].mean():.3f}")
print()
print("Max absolute differences (K):")
print(f"  odc-geo  vs Satpy:   {odc_sat[common].max():.3f}")
print(f"  odc-geo  vs FAISS:   {odc_fai[common].max():.3f}")
print(f"  Satpy    vs FAISS:   {sat_fai[common].max():.3f}")

## 7. Visualise

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

odc_da.plot(ax=axes[0], cmap="viridis", add_labels=False)
axes[0].set_title("odc-geo xr_reproject")

satpy_da.plot(ax=axes[1], cmap="viridis", add_labels=False)
axes[1].set_title("Satpy Scene.resample")

faiss_da.plot(ax=axes[2], cmap="viridis", add_labels=False)
axes[2].set_title("FAISS nearest neighbour")

for ax in axes:
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()

## Take-aways

* `odc-geo`'s `xr_reproject` works on rectilinear sources. For a real VIIRS swath, the cleanest path is to first rectify the 2-D lat/lon data to a regular lat/lon grid (here with `scipy.interpolate.griddata`), then let `xr_reproject` warp that grid to UTM.
* `Satpy.Scene.resample` does the rectification and reprojection in one step via its `AreaDefinition` machinery and is the most convenient option when Satpy is already reading the data.
* FAISS gives explicit nearest-neighbour control and is easy to adapt to GPU (`faiss-gpu`) or approximate indices for very large swaths.

With the intermediate lat/lon rectification, `odc-geo` and FAISS agree to within a fraction of a kelvin, while Satpy differs by a few kelvin because its resampling pipeline uses the sensor geometry rather than a purely geographic nearest-neighbour lookup.